# Practical Session 4: Introduction to Sensing

## Redes de Sensores y Sistemas Autónomos 
### Grado en Ingeniería de las Tecnologías de Telecomunicación
### Universidad de Sevilla

David Alejo Teissière

## Part 1: Laserscan message and Emergency stop node

To this point, our Turtlebot was a little bit sad and a feeling of loneliness embraced her as she explored an empty world. But this will change in this session. From now on, we will let our beloved Turtlebot explore a more interesting environment (see Fig. 1 left). Figure 1 right shows the measurements gathered from the 2D LiDAR sensor due to the obstacles surrounding her.

__Exercise_1__

Copy the ROS package named `collision_avoidance`, which is available in the `practica_4` folder. It depends on the `nav2_amcl`, `geometry_msgs`, `nav2_map_server`, `nav_msgs`, `rclpy`, `sensor_msgs`, `std_msgs` and `tf2_ros` packages.

We use the `practica_4.launch.xml` launch file to start all the necessary nodes for simulation, trajectory tracking, a visualization node and some localization nodes, which will be used to localize the turtlebot in the map. 

Once you have copied the `collision_avoidance` package, please start the simulation by using the following command (note that the first time it can take its time, as it has to download some resources from the Internet).

In [ ]:
(rssa Docker) >  cd ~/rssa_ws
(rssa Docker) >  source install/setup.bash
(rssa Docker) >  colcon build --symlink-install
(rssa Docker) > ros2 launch collision_avoidance practica_4.launch.xml

<figure style="text-align:center">
  <img src="images/turtlebot_house.png" alt="" width=900>
  <figcaption>Fig. 1: (a) Gazebo simulation of our turtlebot in a house environment. (b) Obtained LaserScan of the Turtlebot  </figcaption>
</figure>

If we want to avoid obstacles, we need to use a sensor to detect the obstacles. Particularly, our Turtlebot3 robot includes a 2D laser range finder: https://emanual.robotis.com/docs/en/platform/turtlebot3/appendix_lds_01/ . 
The Gazebo simulator is able to simulate this sensor among many others. Therefore, we can read the 2D laser scan readings, since our simulator is publishing this data. We can see with the tool rqt the publications. In this case we are interested in /scan topic [sensor_msgs/LaserScan]. Fig. 1 (b) represents one measurement of the sensor onboard the Turtlebot.

From a 2D laser scan we obtain a set of distances (ranges) to the obstacles around the sensor. Each distance position in the vector of distances corresponds to a known angle, so we have a set of *polar coordinates* of the obstacles regarding the center of the sensor. 

Thus, the nodes of the `collision_avoidance` package should subscribe the `/scan` topic with type: `sensor_msgs.msg.LaserScan`. The data structure is presented next:

In [ ]:
################ Contents of the LaserScan.msg file of package sensor_msgs ####################################
# Single scan from a planar laser range-finder
#
# If you have another ranging device with different behavior (e.g. a sonar
# array), please find or create a different message, since applications
# will make fairly laser-specific assumptions about this data

std_msgs/Header header            # timestamp in the header is the acquisition time of 
                         # the first ray in the scan.
                         #
                         # in frame frame_id, angles are measured around 
                         # the positive Z axis (counterclockwise, if Z is up)
                         # with zero angle being forward along the x axis
                         
float32 angle_min        # start angle of the scan [rad]
float32 angle_max        # end angle of the scan [rad]
float32 angle_increment  # angular distance between measurements [rad]

float32 time_increment   # time between measurements [seconds] - if your scanner
                         # is moving, this will be used in interpolating position
                         # of 3d points
float32 scan_time        # time between scans [seconds]

float32 range_min        # minimum range value [m]
float32 range_max        # maximum range value [m]

float32[] ranges         # range data [m] (Note: values < range_min or > range_max should be discarded)
float32[] intensities    # intensity data [device-specific units].  If your
                         # device does not provide intensities, please leave
                         # the array empty.


In this class, you will develop at least two scripts in your `collision_avoidance` package. 

The idea of any node inside this package is that it susbscribes to the `v_pref` topic (preferred velocity, of type `geometry_msgs/msg/TwistStamped`), which usually comes from the path tracker node. Then, taking into account the obstacles sensed by the laser sensor, it generates a collision-free velocity as close as possible to the desired and publishes it in the `cmd_vel` topic (same type as `v_pref`), which is received and executed by the Turtlebot itself. This idea is depicted in Fig. 2.

<figure style="text-align:center">
  <img src="images/diagram_collision_avoidance.png" alt="" width=600>
  <figcaption>Fig. 2: (a) Proposed block diagram of the Collision Avoidance system. </figcaption>
</figure>

__Exercise_2__

In a first approximation to the collision avoidance method, we will implement a basic node that subscribes the scan and stops the Turtlebot if an obstacle is detected with a distance minor than a given threshold `min_d`. This distance should be configurable by means of the parameter server.

Thus, modify the node `emergency_stop.py` which subscribes to the `v_pref` and the `scan` topics. Then each time a scan is received it moves to a emergency_stop state if a threat is detected.

The node should periodically publish in the `cmd_vel` topic the last `v_pref` received under normal circumstances or zero velocity (note that only linear velocity should be modified) if a threat has been detected.

Add a new node tag in the `practica_4.launch.xml` file that includes your newly created `emergency_stop.py` node and test that it works properly.

## Part 2: Implementation of a reactive Collision Avoidance method

Please implement one of the following collision avoidance method so that our robot is able to circumvent unmodeled obstacles.

- The bug algorithm (first recommendation)
- Artificial potential fields
- Methods based on velocity obstacles (such as the Optimal Reciprocal Collision Avoidance, ORCA)
- Dynamic Window Approach (DWA) (second recommendation)
- Time-elastic band

__Exercise_3__ (optional)

With the implemented `emergency_stop.py` as a base, implement the bug algorithm that will rotate the robot when emergency stop state until no obstacle is found at front.

Then, it will move forward a determinate number of steps, before returning to the initial follow trajectory state. From this state, if an obstacle is found then we can pass to the rotating state again, or if the number of steps has been reached.

Therefore, it will be a state machine with three different states: following, rotating and moving forward.

__Exercise_4__

Modify the `DWA.py` node to implement the dynamic window approach seen on class. It should share the same interface as the node of Exercise 1. Add as many parameters of the node so that its behavior can be configured with the ROS parameter server. Added the corresponding entry point to `setup.py`.

In the `practica_4.launch.xml` launch file, replace the previously generated `emergency_stop.py` node with your `DWA.py` node. Test the algorithm under different circumstances